# FlowGraphFormer
## 00B_Dataset_Harmonizer.ipynb

Run this **once**, before `01_Preprocessing_local.ipynb`, whenever you add a new
CIC-IDS release (2017, 2018, 2019, 2023, ...) into `dataset/raw/`.

**What it does, and why:**

`01_Preprocessing_local.ipynb` (Cell 3) does `pd.concat()` on every CSV in
`dataset/raw/`. If the CSVs come from different CIC-IDS years, they will
almost certainly have different column names, different feature counts,
and inconsistent attack-label strings. `pd.concat` will NOT error on this
-- it silently unions the columns and fills mismatches with `NaN`, which
corrupts cleaning, feature selection, and SMOTE downstream without ever
raising a warning you'd notice.

This notebook fixes that **before** it reaches Preprocessing:

1. Backs up every original raw CSV to `dataset/raw_backup/` (never destructive).
2. Detects the column schema of every CSV in `dataset/raw/`.
3. Maps every file's columns onto one **canonical schema** (known CIC-IDS
   naming aliases + fuzzy matching for anything not in the known list).
4. Drops identifier/leakage-prone columns (Flow ID, Source IP, Source
   Port, Timestamp) that make a model learn "which capture this came
   from" instead of real attack behavior.
5. Standardizes the `Label` column: strips whitespace, and groups messy
   variant attack names (`"DDoS attack-HOIC"`, `"DDOS-HOIC"`, ...) into
   one consistent taxonomy.
6. Overwrites each file in `dataset/raw/` with its harmonized version --
   same filenames, so `01_Preprocessing_local.ipynb` needs **zero code changes**.
7. Saves a full report to `reports/harmonization/` +
   `.json` so you can see exactly what was mapped, dropped, or flagged
   for manual review.

**After this runs successfully, just run `01_Preprocessing_local.ipynb` as-is.**


In [ ]:
# ==============================================================================
# FlowGraphFormer
# 00B_Dataset_Harmonizer.ipynb
#
# Cell 0 : Environment Detection & Initialization (Colab + Local)
# ==============================================================================

import os
import sys
import gc
import re
import json
import shutil
import difflib
import warnings
import subprocess
import importlib
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

print("=" * 80)
print("FlowGraphFormer")
print("Research Notebook : 00B_Dataset_Harmonizer")
print("=" * 80)

# ------------------------------------------------------------------------------
# Detect Runtime (Colab vs Local)
# ------------------------------------------------------------------------------

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

print(f"Runtime Detected      : {'Google Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

PROJECT_NAME = "FlowGraphFormer"

if IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/MyDrive/ML_Project") / PROJECT_NAME
else:
    _cwd = Path.cwd().resolve()
    if _cwd.name == PROJECT_NAME:
        PROJECT_ROOT = _cwd
    elif (_cwd / PROJECT_NAME).exists():
        PROJECT_ROOT = _cwd / PROJECT_NAME
    else:
        PROJECT_ROOT = _cwd

print(f"Project Root          : {PROJECT_ROOT}")

# ------------------------------------------------------------------------------
# Install Missing Packages
# ------------------------------------------------------------------------------

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "tqdm": "tqdm",
}

missing_packages = []
for module, package in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module)
    except ImportError:
        missing_packages.append(package)

if missing_packages:
    print(f"\nInstalling missing packages: {missing_packages}\n")
    pip_cmd = [sys.executable, "-m", "pip", "install", "-q"]
    if not IN_COLAB:
        pip_cmd.append("--break-system-packages")
    subprocess.check_call(pip_cmd + missing_packages)
else:
    print("\nAll required packages already installed.\n")

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# ------------------------------------------------------------------------------
# Directories (same layout as 01_Preprocessing_local.ipynb)
# ------------------------------------------------------------------------------

DATASET_DIR = PROJECT_ROOT / "dataset"
RAW_DATASET_DIR = DATASET_DIR / "raw"
RAW_BACKUP_DIR = DATASET_DIR / "raw_backup"
REPORT_DIR = PROJECT_ROOT / "reports" / "harmonization"

for directory in [RAW_DATASET_DIR, RAW_BACKUP_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Raw Dataset Directory  : {RAW_DATASET_DIR}")
print(f"Backup Directory       : {RAW_BACKUP_DIR}")
print(f"Report Directory       : {REPORT_DIR}")
print("\nNotebook Ready.")
print("=" * 80)


In [ ]:
# ==============================================================================
# Cell 1 : Configuration
# ==============================================================================

CONFIG = {

    # If a column can't be matched to the canonical schema (by alias or
    # fuzzy match) with at least this similarity, it's dropped and logged
    # rather than guessed at.
    "fuzzy_match_threshold": 0.82,

    # Columns to always drop if present -- these identify WHERE/WHEN a
    # flow was captured rather than WHAT the flow looked like, and are a
    # classic shortcut-learning trap when merging multiple capture
    # environments (different CIC-IDS years = different networks).
    "leakage_columns": [
        "flow_id",
        "source_ip", "src_ip",
        "source_port", "src_port",
        "destination_ip", "dst_ip",
        "timestamp",
        "unnamed:_0",  # pandas index column some CIC CSVs ship with
        "similarhttp",  # known junk column in some 2018/2019 releases
    ],

    # Which raw file's columns to treat as the canonical/reference
    # schema. "auto" picks the file with "2017" in its name if one
    # exists, otherwise the largest file by column count.
    "reference_file_hint": "2017",

    # If a canonical column is completely missing from a given file,
    # it's filled with NaN so the row shape stays consistent -- your
    # existing Cell 6 (Data Cleaning) in 01_Preprocessing_local.ipynb already
    # handles missing-value imputation, so this is safe rather than
    # silently dropping the whole file.
    "fill_missing_columns_with_nan": True,

    # Overwrite files in dataset/raw/ in place (recommended -- keeps
    # 01_Preprocessing_local.ipynb completely unchanged). Originals are always
    # backed up to dataset/raw_backup/ first regardless of this setting.
    "overwrite_in_place": True,

}

print("Configuration loaded.")
print(json.dumps({k: v for k, v in CONFIG.items() if k != "leakage_columns"}, indent=2))


In [ ]:
# ==============================================================================
# Cell 2 : Column Alias Dictionary & Label Taxonomy
# ==============================================================================
# CICFlowMeter output column names differ across CIC-IDS releases -- most
# notably CSE-CIC-IDS2018 abbreviates almost every column name relative to
# CIC-IDS2017 (e.g. "Total Fwd Packets" -> "Tot Fwd Pkts"). This is a
# known, documented naming difference, not a data quality issue.
#
# Keys and values below are already normalized the same way column names
# get normalized later (lowercase, spaces/slashes -> underscore) so the
# lookup is a direct dict hit after normalization.

def normalize_name(name):
    name = str(name).strip().lower()
    name = name.replace("/", "_").replace(" ", "_")
    name = re.sub(r"_+", "_", name)
    return name.strip("_")


# Known CIC-IDS2017 <-> CSE-CIC-IDS2018 (and common variant) aliases.
# Canonical form (dict value) follows CIC-IDS2017 naming, since that is
# what 01_Preprocessing_local.ipynb / the rest of this project was built around.
COLUMN_ALIASES = {
    "dst_port": "destination_port",
    "tot_fwd_pkts": "total_fwd_packets",
    "tot_bwd_pkts": "total_backward_packets",
    "totlen_fwd_pkts": "total_length_of_fwd_packets",
    "totlen_bwd_pkts": "total_length_of_bwd_packets",
    "fwd_pkt_len_max": "fwd_packet_length_max",
    "fwd_pkt_len_min": "fwd_packet_length_min",
    "fwd_pkt_len_mean": "fwd_packet_length_mean",
    "fwd_pkt_len_std": "fwd_packet_length_std",
    "bwd_pkt_len_max": "bwd_packet_length_max",
    "bwd_pkt_len_min": "bwd_packet_length_min",
    "bwd_pkt_len_mean": "bwd_packet_length_mean",
    "bwd_pkt_len_std": "bwd_packet_length_std",
    "flow_byts_s": "flow_bytes_s",
    "flow_pkts_s": "flow_packets_s",
    "fwd_iat_tot": "fwd_iat_total",
    "bwd_iat_tot": "bwd_iat_total",
    "fwd_header_len": "fwd_header_length",
    "bwd_header_len": "bwd_header_length",
    "fwd_pkts_s": "fwd_packets_s",
    "bwd_pkts_s": "bwd_packets_s",
    "pkt_len_min": "min_packet_length",
    "pkt_len_max": "max_packet_length",
    "pkt_len_mean": "packet_length_mean",
    "pkt_len_std": "packet_length_std",
    "pkt_len_var": "packet_length_variance",
    "fin_flag_cnt": "fin_flag_count",
    "syn_flag_cnt": "syn_flag_count",
    "rst_flag_cnt": "rst_flag_count",
    "psh_flag_cnt": "psh_flag_count",
    "ack_flag_cnt": "ack_flag_count",
    "urg_flag_cnt": "urg_flag_count",
    "cwe_flag_count": "cwe_flag_count",
    "cwr_flag_count": "cwe_flag_count",
    "ece_flag_cnt": "ece_flag_count",
    "pkt_size_avg": "average_packet_size",
    "fwd_seg_size_avg": "avg_fwd_segment_size",
    "bwd_seg_size_avg": "avg_bwd_segment_size",
    "fwd_byts_b_avg": "fwd_avg_bytes_bulk",
    "fwd_pkts_b_avg": "fwd_avg_packets_bulk",
    "fwd_blk_rate_avg": "fwd_avg_bulk_rate",
    "bwd_byts_b_avg": "bwd_avg_bytes_bulk",
    "bwd_pkts_b_avg": "bwd_avg_packets_bulk",
    "bwd_blk_rate_avg": "bwd_avg_bulk_rate",
    "subflow_fwd_pkts": "subflow_fwd_packets",
    "subflow_fwd_byts": "subflow_fwd_bytes",
    "subflow_bwd_pkts": "subflow_bwd_packets",
    "subflow_bwd_byts": "subflow_bwd_bytes",
    "init_fwd_win_byts": "init_win_bytes_forward",
    "init_bwd_win_byts": "init_win_bytes_backward",
    "fwd_act_data_pkts": "act_data_pkt_fwd",
    "fwd_seg_size_min": "min_seg_size_forward",
    "protocol": "protocol",
    "flow_duration": "flow_duration",
    "label": "label",
}

# Leakage/identifier columns that should never be treated as a "missing
# canonical feature" even if they show up under a slightly different name.
LEAKAGE_ALIASES = set(normalize_name(c) for c in CONFIG["leakage_columns"])


def resolve_column(col, canonical_columns, threshold):
    """Map one (already-normalized) column name onto the canonical schema.
    Returns (resolved_name_or_None, method) where method is one of:
    'exact', 'alias', 'fuzzy', 'leakage', 'unmatched'."""

    if col in LEAKAGE_ALIASES:
        return None, "leakage"

    if col in canonical_columns:
        return col, "exact"

    if col in COLUMN_ALIASES and COLUMN_ALIASES[col] in canonical_columns:
        return COLUMN_ALIASES[col], "alias"

    # Reverse-alias: canonical schema itself might be the abbreviated form
    for variant, canon in COLUMN_ALIASES.items():
        if canon == col and variant in canonical_columns:
            return variant, "alias"

    match = difflib.get_close_matches(col, canonical_columns, n=1, cutoff=threshold)

    if match:
        return match[0], "fuzzy"

    return None, "unmatched"


# ==============================================================================
# Label Taxonomy (keyword-based, order matters -- first match wins)
# ==============================================================================

LABEL_RULES = [

    (["benign", "normal"], "BENIGN"),

    (["heartbleed"], "Heartbleed"),

    (["ddos", "distributed denial"], "DDoS"),

    (["dos slowloris", "slowloris"], "DoS"),

    (["dos slowhttptest", "slowhttptest"], "DoS"),

    (["dos hulk", "hulk"], "DoS"),

    (["dos goldeneye", "goldeneye"], "DoS"),

    (["dos "], "DoS"),

    (["portscan", "port scan"], "PortScan"),

    (["botnet", "bot"], "Bot"),

    (["infiltration"], "Infiltration"),

    (["patator", "brute force", "bruteforce", "brute-force"], "BruteForce"),

    (["sql injection", "sqli"], "WebAttack"),

    (["xss", "cross site"], "WebAttack"),

    (["web attack", "webattack"], "WebAttack"),

]


def canonicalize_label(raw_label):
    """Maps a raw label string to the unified taxonomy. Anything that
    doesn't match a known keyword is kept as-is but flagged for the
    report, rather than silently discarded."""

    if pd.isna(raw_label):
        return None, "missing"

    cleaned = str(raw_label).strip()
    lowered = cleaned.lower()

    for keywords, canonical in LABEL_RULES:
        if any(kw in lowered for kw in keywords):
            return canonical, "mapped"

    return cleaned, "unmapped"


print("Column alias dictionary  :", len(COLUMN_ALIASES), "known aliases")
print("Label taxonomy rules     :", len(LABEL_RULES), "keyword groups")


In [ ]:
# ==============================================================================
# Cell 3 : Detect Canonical Reference Schema
# ==============================================================================

print("=" * 80)
print("Scanning dataset/raw/ for CSV files")
print("=" * 80)

csv_files = sorted(RAW_DATASET_DIR.glob("*.csv"))

if len(csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in:\n{RAW_DATASET_DIR}")

print(f"Found {len(csv_files)} CSV file(s):\n")
for f in csv_files:
    print(" -", f.name)

# ==============================================================================
# Read just the header row of every file (fast -- no full load needed)
# ==============================================================================

file_columns = {}

for file in tqdm(csv_files, desc="Reading headers"):

    header = pd.read_csv(file, nrows=0)

    normalized_cols = [normalize_name(c) for c in header.columns]

    file_columns[file.name] = normalized_cols

# ==============================================================================
# Pick the Reference File
# ==============================================================================

reference_file = None

for file in csv_files:
    if CONFIG["reference_file_hint"].lower() in file.name.lower():
        reference_file = file
        break

if reference_file is None:
    # Fall back to whichever file has the most columns (most complete schema)
    reference_file = max(csv_files, key=lambda f: len(file_columns[f.name]))

CANONICAL_COLUMNS = [
    c for c in file_columns[reference_file.name]
    if c not in LEAKAGE_ALIASES
]

print("\n" + "=" * 80)
print(f"Reference File     : {reference_file.name}")
print(f"Canonical Features : {len(CANONICAL_COLUMNS)} (after dropping leakage columns)")
print("=" * 80)

if "label" not in CANONICAL_COLUMNS:
    raise ValueError(
        f"Reference file {reference_file.name} has no recognizable 'Label' "
        f"column after normalization -- cannot proceed. Check the file "
        f"manually or set CONFIG['reference_file_hint'] to a different file."
    )


In [ ]:
# ==============================================================================
# Cell 4 : Harmonize Every File and Save Back to dataset/raw/
# ==============================================================================

print("=" * 80)
print("Harmonizing All Raw CSV Files")
print("=" * 80)

file_reports = []
label_value_counts_by_file = {}

for file in tqdm(csv_files, desc="Harmonizing"):

    df = pd.read_csv(file, low_memory=False)

    original_shape = df.shape

    df.columns = [normalize_name(c) for c in df.columns]

    # --------------------------------------------------------------------
    # Resolve every column onto the canonical schema
    # --------------------------------------------------------------------

    resolved_map = {}      # original_col -> canonical_col
    unmatched_cols = []
    leakage_cols_found = []
    method_counts = {"exact": 0, "alias": 0, "fuzzy": 0}

    for col in df.columns:

        if col == "label":
            continue

        resolved, method = resolve_column(
            col,
            [c for c in CANONICAL_COLUMNS if c != "label"],
            CONFIG["fuzzy_match_threshold"],
        )

        if method == "leakage":
            leakage_cols_found.append(col)
        elif resolved is None:
            unmatched_cols.append(col)
        else:
            resolved_map[col] = resolved
            method_counts[method] += 1

    # --------------------------------------------------------------------
    # Build the harmonized frame: canonical columns only, correctly named
    # --------------------------------------------------------------------

    harmonized = pd.DataFrame(index=df.index)

    for original_col, canonical_col in resolved_map.items():
        harmonized[canonical_col] = df[original_col]

    missing_canonical = [
        c for c in CANONICAL_COLUMNS
        if c != "label" and c not in harmonized.columns
    ]

    if CONFIG["fill_missing_columns_with_nan"]:
        for c in missing_canonical:
            harmonized[c] = np.nan

    # Reorder to match the canonical schema exactly (features first, label last)
    ordered_cols = [c for c in CANONICAL_COLUMNS if c != "label" and c in harmonized.columns]
    harmonized = harmonized[ordered_cols]

    # --------------------------------------------------------------------
    # Standardize the Label column
    # --------------------------------------------------------------------

    raw_labels = df["label"] if "label" in df.columns else pd.Series([np.nan] * len(df))

    canonical_labels, label_methods = [], []

    for raw in raw_labels:
        canon, method = canonicalize_label(raw)
        canonical_labels.append(canon)
        label_methods.append(method)

    harmonized["Label"] = canonical_labels

    n_missing_labels = sum(m == "missing" for m in label_methods)
    n_unmapped_labels = sum(m == "unmapped" for m in label_methods)

    if n_missing_labels:
        before = len(harmonized)
        harmonized = harmonized[harmonized["Label"].notna()]
        print(f"  {file.name}: dropped {before - len(harmonized)} rows with missing Label")

    label_value_counts_by_file[file.name] = harmonized["Label"].value_counts().to_dict()

    # --------------------------------------------------------------------
    # Coerce feature columns to numeric (protects against stray strings /
    # inf-as-string values that sometimes appear in raw CIC-IDS CSVs)
    # --------------------------------------------------------------------

    for c in ordered_cols:
        harmonized[c] = pd.to_numeric(harmonized[c], errors="coerce")

    harmonized.replace([np.inf, -np.inf], np.nan, inplace=True)

    # --------------------------------------------------------------------
    # Backup original, then overwrite in place
    # --------------------------------------------------------------------

    backup_path = RAW_BACKUP_DIR / file.name

    if not backup_path.exists():
        shutil.copy2(file, backup_path)

    if CONFIG["overwrite_in_place"]:
        harmonized.to_csv(file, index=False)

    file_reports.append({
        "File": file.name,
        "Original Rows": original_shape[0],
        "Original Columns": original_shape[1],
        "Harmonized Rows": len(harmonized),
        "Harmonized Columns": harmonized.shape[1],
        "Exact Matches": method_counts["exact"],
        "Alias Matches": method_counts["alias"],
        "Fuzzy Matches": method_counts["fuzzy"],
        "Unmatched (Dropped)": len(unmatched_cols),
        "Unmatched Columns": ", ".join(unmatched_cols) if unmatched_cols else "-",
        "Leakage Columns Dropped": len(leakage_cols_found),
        "Missing Canonical Cols (NaN-filled)": len(missing_canonical),
        "Labels Missing (Dropped)": n_missing_labels,
        "Labels Unmapped (Kept As-Is)": n_unmapped_labels,
    })

    del df, harmonized
    gc.collect()

file_report_df = pd.DataFrame(file_reports)

print("\n" + "=" * 80)
print("Per-File Harmonization Report")
print("=" * 80)
display(file_report_df)


In [ ]:
# ==============================================================================
# Cell 5 : Save Reports & Verify Compatibility
# ==============================================================================

print("=" * 80)
print("Saving Reports")
print("=" * 80)

# ==============================================================================
# Per-File Report (CSV)
# ==============================================================================

file_report_df.to_csv(
    REPORT_DIR / "harmonization_report.csv",
    index=False,
)

# ==============================================================================
# Label Distribution Across All Files (helps spot dataset-specific skew)
# ==============================================================================

all_labels = sorted({
    label
    for counts in label_value_counts_by_file.values()
    for label in counts
})

label_matrix = pd.DataFrame(0, index=all_labels, columns=list(label_value_counts_by_file.keys()))

for file_name, counts in label_value_counts_by_file.items():
    for label, count in counts.items():
        label_matrix.loc[label, file_name] = count

label_matrix["Total"] = label_matrix.sum(axis=1)
label_matrix = label_matrix.sort_values("Total", ascending=False)

label_matrix.to_csv(REPORT_DIR / "label_distribution_by_file.csv")

print("\nUnified Label Distribution (rows = canonical label, columns = source file):")
display(label_matrix)

# ==============================================================================
# Full JSON Summary
# ==============================================================================

summary = {
    "generated_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "reference_file": reference_file.name,
    "canonical_feature_count": len(CANONICAL_COLUMNS) - 1,  # exclude label
    "files_processed": len(csv_files),
    "total_rows_after_harmonization": int(file_report_df["Harmonized Rows"].sum()),
    "unified_label_categories": all_labels,
    "config": {k: v for k, v in CONFIG.items() if k != "leakage_columns"},
}

with open(REPORT_DIR / "harmonization_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

# ==============================================================================
# Post-Harmonization Compatibility Check
# ==============================================================================
# Re-reads just the headers of the now-overwritten files in dataset/raw/
# to confirm every file has an IDENTICAL column set -- i.e. that
# 01_Preprocessing_local.ipynb's pd.concat() will merge cleanly with no
# silent NaN-filling from mismatched schemas.

print("\n" + "=" * 80)
print("Post-Harmonization Compatibility Check")
print("=" * 80)

post_columns = {}
for file in csv_files:
    header = pd.read_csv(file, nrows=0)
    post_columns[file.name] = tuple(sorted(header.columns.tolist()))

unique_schemas = set(post_columns.values())

if len(unique_schemas) == 1:
    print("PASS -- all files in dataset/raw/ now share an identical schema.")
    print(f"Columns per file : {len(list(unique_schemas)[0])}")
else:
    print("FAIL -- files still have differing schemas after harmonization:")
    for file_name, cols in post_columns.items():
        print(f"  {file_name}: {len(cols)} columns")
    print("\nInspect harmonization_report.csv for unmatched/dropped columns per file.")

print("\n" + "=" * 80)
print("Reports Saved:")
print(REPORT_DIR / "harmonization_report.csv")
print(REPORT_DIR / "label_distribution_by_file.csv")
print(REPORT_DIR / "harmonization_summary.json")
print("=" * 80)
print("\nOriginal files backed up to:")
print(RAW_BACKUP_DIR)
print("\ndataset/raw/ now contains harmonized, schema-consistent CSVs.")
print("You can run 01_Preprocessing_local.ipynb next with NO code changes.")
print("=" * 80)
